# 02 · Ingest CFBD data: betting lines, advanced stats, talent, returning production

Source: CollegeFootballData (CFBD). Needs `CFBD_API_KEY` in `~/.zshrc`; the key is read
from the environment and never written anywhere.

The free tier has a monthly call limit (3,000), so every response is cached in
`data/raw/cfbd/`. A full rebuild of 2015–2026 costs ~60 calls; re-runs cost 0 except for
the current season, which is refreshed.

CFBD game ids are the same as ESPN's, so everything joins on `game_id`.

In [ ]:
import os

import pandas as pd

from canes_cfb import cfbd
from canes_cfb.paths import RAW

assert os.environ.get("CFBD_API_KEY"), "Set CFBD_API_KEY in ~/.zshrc and restart Jupyter"
SEASONS = list(range(2015, 2027))
CURRENT = 2026

## 1. Betting lines

Consensus = median across books (ESPN Bet, DraftKings, Bovada...). Spreads are the home
team's spread (negative = home favored), the convention in `docs/markets.md`.

In [ ]:
lines = cfbd.load_lines(SEASONS, current_season=CURRENT)
lines.to_parquet(RAW / "lines.parquet", index=False)
lines.head()

## 2. Advanced game stats (per team-game: plays, EPA/PPA, success rate, explosiveness, pass/rush splits)

In [ ]:
advanced = cfbd.load_advanced(SEASONS, current_season=CURRENT)
advanced.to_parquet(RAW / "advanced.parquet", index=False)
advanced.shape

## 3. Season priors (talent composite, returning production) and team ids

In [ ]:
cfbd.load_talent(SEASONS).to_parquet(RAW / "talent.parquet", index=False)
cfbd.load_returning(SEASONS).to_parquet(RAW / "returning.parquet", index=False)
cfbd.load_teams().to_parquet(RAW / "teams.parquet", index=False)

## 4. Coverage against ESPN games

In [ ]:
games = pd.read_parquet(RAW / "games.parquet")
fbs = games[games.completed & games.home_fbs & games.away_fbs & (games.season <= 2025)]
coverage = pd.DataFrame(
    {
        "closing spread": fbs.merge(lines, on="game_id", how="left")
        .groupby("season")
        .spread_close.apply(lambda s: s.notna().mean()),
        "opening spread": fbs.merge(lines, on="game_id", how="left")
        .groupby("season")
        .spread_open.apply(lambda s: s.notna().mean()),
        "advanced stats": fbs.assign(has=fbs.game_id.isin(advanced.game_id))
        .groupby("season")
        .has.mean(),
    }
)
(coverage * 100).round(1)

Closing lines cover every FBS game; opening lines only ~64% (not every book reports
them). Advanced stats cover 99.6%.